# WEEK5 — RAG Live (Student Version)

A compact student-friendly notebook with only headings and hint-only code cells.

## Step 1 — Setup

In [2]:
# Hint: load the .env file with load_dotenv() and point Path.cwd() at the notebook folder
# Hint: set LANGSMITH_TRACING / LANGCHAIN_TRACING_V2 env vars so LangSmith can capture the session
# Hint: print a quick diagnostic - cwd, whether .env loaded, whether OPENAI/LANGSMITH keys are present, and which PDF files are in the folder

from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()
print('CWD:', Path.cwd())
print('OPENAI_API_KEY present:', bool(os.getenv('OPENAI_API_KEY')))
print('PDF files in folder:', [p.name for p in sorted(Path.cwd().glob('*.pdf'))])

CWD: d:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 5 — RAG Fundamentals (Retrieval-Augmented Generation)\Live
OPENAI_API_KEY present: True
PDF files in folder: ['HDFC-Life-Group-Term-Life-Policy.pdf', 'HDFC-Life-Sampoorna-Jeevan-101N158V04-Policy-Document (1).pdf', 'HDFC-Life-Sanchay-Plus-Life-Long-Income-Option-101N134V19-Policy-Document.pdf']


In [ ]:
# Hint: load environment variables (e.g., OPENAI_API_KEY)
# Hint: import necessary libraries: langchain/text_splitters, vectorstores, openai client, numpy
# Hint: ensure your Python kernel has required packages installed


## Step 2 — Load PDF documents

In [4]:
# Hint: list PDF files in the data folder
# Hint: use a PDF loader to extract text into a list of documents
# Hint: verify you have 3 PDF files and show filenames to students

from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

pdf_files = sorted(Path.cwd().glob('*.pdf'))
docs = []
for pdf_file in pdf_files:
    docs.extend(PyPDFLoader(str(pdf_file)).load())

print('PDF files:', [p.name for p in pdf_files])
print('Loaded pages:', len(docs))
print('First page preview:')
print(docs[0].page_content[:800].replace('\n', ' '))

PDF files: ['HDFC-Life-Group-Term-Life-Policy.pdf', 'HDFC-Life-Sampoorna-Jeevan-101N158V04-Policy-Document (1).pdf', 'HDFC-Life-Sanchay-Plus-Life-Long-Income-Option-101N134V19-Policy-Document.pdf']
Loaded pages: 101
First page preview:
F&U dated 15th October 2022                  UIN-101N169V02  P a g e  | 0                                      HDFC Life Group Term Life    OF      «OWNERNAME»               Based on the Proposal and the declarations and  any  statement made or referred to therein,  We will pay the Benefits mentioned in this Policy  subject to the terms and conditions contained  herein              << Designation of the Authorised Signatory >>


In [5]:
docs

[Document(metadata={'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': '2023-08-24T19:47:11+05:30', 'title': 'Exide Life Group Term Life (UIN 114N012V03) – Terms and Conditions', 'author': 'Atul Bhatia', 'moddate': '2023-08-24T19:47:11+05:30', 'source': 'd:\\Mentoring\\learwithsarvesh\\AI Engineer Ready\\WEEK 5 — RAG Fundamentals (Retrieval-Augmented Generation)\\Live\\HDFC-Life-Group-Term-Life-Policy.pdf', 'total_pages': 30, 'page': 0, 'page_label': '1'}, page_content='F&U dated 15th October 2022                  UIN-101N169V02  P a g e  | 0                        \n \n \n \n \n \n   HDFC Life Group Term Life \n \nOF \n \n \n«OWNERNAME» \n \n \n \n \n \n  \nBased on the Proposal and the declarations and \nany \nstatement made or referred to therein, \nWe will pay the Benefits mentioned in this Policy \nsubject to the terms and conditions contained \nherein \n \n \n \n \n \n \n<< Designation of the Authorised Signatory >>'),
 Document(m

In [ ]:
# Hint: inspect the first loaded document - check its page_content and metadata


In [ ]:
# Hint: display the list of PDF files that were discovered


## Step 3 — Chunking (brief)

Context Window: Is the amount of information an AI model can "see" or "remember" at once while generating a response;

- Chunking: Breaking large text into smaller, meaningful pieces (chunks)
Why do we do this?
1. LLMs cannot process very large documents at once
2. Embeddings work best when the text is focussed and semantically tight

### Without Chunking:
1. Large Docuemnts
2. Hard to search
3. Token cost is higher
4. Poor accuracy/ Emdedding Poor

### With Chunking:
1. Small pieces
2. Easy Retrieval
3. Better matching
4. Lower Cost

Recursive Character Text Splitter:

- Without breaking meaning
- While respecting the structure (Paragraphs -> Sentences -> Words -> chars)

- Recursively tries different seperators until it finds a good split

1. Try splitting with first seperator (\n\n)
2. If chunk is too big -> goto the next separator (\n)
3. chunk size <= limit or the last separator

"This policy matu" | "res on 07th June 2026"

"This policy matures on 07th June 2026" #Keep the semantic intact

- 900 - Paragraph ends (Seperate Chunk)
If no paragraph :
- 950 - A new line (Seperate Chunk)
- 980 - A new word (Seperate chunk)

We control chunking behaviour by designing separators

RecursiveCharacterTextSplitter -> It has a hierarchy of seperators.
- \n\n -> Paragraph
- \n -> line
- " " -> word
- "" -> Individual characters

In [6]:
# Hint: choose a chunking strategy (fixed chars / sentences / token-based)
# Hint: show students the chosen chunk size and overlap
# Hint: print a single example chunk for verification

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 150,
)

chunks = splitter.split_documents(docs)

print("Chunks created:", len(chunks))
print("First chunk preview:")
print(chunks[0].page_content[:800].replace('\n', " "))

Chunks created: 324
First chunk preview:
F&U dated 15th October 2022                  UIN-101N169V02  P a g e  | 0                                      HDFC Life Group Term Life    OF      «OWNERNAME»               Based on the Proposal and the declarations and  any  statement made or referred to therein,  We will pay the Benefits mentioned in this Policy  subject to the terms and conditions contained  herein              << Designation of the Authorised Signatory >>


## Step 4 — Embeddings & Vector Store

Embeddings: Converting texts into numbers that capture meaning  (Semantics)

We need to search based on meaning, not keywords

In [ ]:
model = 

In [9]:
# Hint: pick an embedding source (OpenAI or local SBERT)
# Hint: embed a small set of chunks and show vector dimension
# Hint: if using FAISS/Chroma, mention dtype=float32 requirement

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.messages import SystemMessage, HumanMessage

embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding=embeddings,
    persist_directory='rag_chroma_store',
)

retriever = vectorstore.as_retriever(search_kwargs={'k':4})

print("Vector Store Ready:", len(chunks), 'chunks')

Vector Store Ready: 324 chunks


## Step 5 — Retriever (simple demo)

Cosine Similarity -> CS measures how similar two vectors are based on their angle

- Not Distance
- Not magnitude
- Only direction (angle)

CS -> Core idea -> Find the meaning

- Same Direction -> Very Similar
- Opposite Direction -> Very Different
- 90 degrees -> Unrelated

- 1 -> Exactly the same meaning
- 0.5 -> Somewhat related
- 0 -> No relation
- -1 -> Opposite Meaning

In [11]:
# Hint: define a sample question to test retrieval against the vector store

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(model = 'gpt-4o-mini', temperature=0)

question = 'What benefits are described in the documents?'
context_docs = retriever.invoke(question)

context_text = '\n\n'.join(  
    f"Source: {doc.metadata.get('source', 'unknown')}\n{doc.page_content}"
    for doc in context_docs
)


messages = [
    SystemMessage(content = "You are a helpful tutor. Answer only from the provided context and keep the answer clear and student-friendly."),
    HumanMessage(content=f'Context:\n{context_text}\n\nQuestion: {question}\n\nAnswer with short citations to the source file names.')
]

response = llm.invoke(messages)
print('Retrieved Context Preview:')

for doc in context_docs:
    source_name = doc.metadata.get("source", "unknown")
    preview = doc.page_content[:220].replace("\n", "")
    print('-', source_name)
    print(' ', preview + ('...' if len(doc.page_content) > 220 else ''))
    print()

print("Final Answer:")
print(response.content)

Retrieved Context Preview:
- d:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 5 — RAG Fundamentals (Retrieval-Augmented Generation)\Live\HDFC-Life-Group-Term-Life-Policy.pdf
   Premium Receipt   :  Acknowledgement of the first Premium paid by you  Terms & Conditions  :  Detailed terms of your Policy contract with HDFC Life                                                                    ...

- d:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 5 — RAG Fundamentals (Retrieval-Augmented Generation)\Live\HDFC-Life-Group-Term-Life-Policy.pdf
   Premium Receipt   :  Acknowledgement of the first Premium paid by you  Terms & Conditions  :  Detailed terms of your Policy contract with HDFC Life                                                                    ...

- d:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 5 — RAG Fundamentals (Retrieval-Augmented Generation)\Live\HDFC-Life-Group-Term-Life-Policy.pdf
   Premium Receipt   :  Acknowledgement of the first Premium paid by you

### Retrieve the best chunks

In [14]:
# Hint: use the retriever to fetch the top-k chunks for the question
# Hint: build a single context string from the retrieved chunks (include source filenames)

from langchain_core.messages import SystemMessage, HumanMessage

test_questions = [
        'What benefits are described in the documents?',
        'What is the policy term or duration?',
        #'What happens on maturity or death?',
    ]

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

for question in test_questions:
    print('='*90)
    print("Question", question)

    context_docs = retriever.invoke(question)
    context_text = '\n\n'.join(
                f"Source: {doc.metadata.get('source', 'unknown')}\n{doc.page_content}"
                for doc in context_docs
            )
    messages = [
                SystemMessage(content='You are a helpful tutor. Answer only from the provided context and keep the answer clear and student-friendly.'),
                HumanMessage(content=f'Context:\n{context_text}\n\nQuestion: {question}\n\nAnswer with short citations to the source file names.'),
            ]

    response = llm.invoke(messages)

    # show retrieved context preview
    print('Retrieved context preview:')
    for doc in context_docs:
        source_name = doc.metadata.get('source', 'unknown')
        preview = doc.page_content[:220].replace('\n', ' ')
        print('-', source_name)
        print(' ', preview + ('...' if len(doc.page_content) > 220 else ''))
        print()

    # show final answer
    print('Final answer:')
    print(response.content)

Question What benefits are described in the documents?
Retrieved context preview:
- d:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 5 — RAG Fundamentals (Retrieval-Augmented Generation)\Live\HDFC-Life-Group-Term-Life-Policy.pdf
   Premium Receipt   :  Acknowledgement of the first Premium paid by you   Terms & Conditions  :  Detailed terms of your Policy contract with HDFC Life                                                                     ...

- d:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 5 — RAG Fundamentals (Retrieval-Augmented Generation)\Live\HDFC-Life-Group-Term-Life-Policy.pdf
   Premium Receipt   :  Acknowledgement of the first Premium paid by you   Terms & Conditions  :  Detailed terms of your Policy contract with HDFC Life                                                                     ...

- d:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 5 — RAG Fundamentals (Retrieval-Augmented Generation)\Live\HDFC-Life-Group-Term-Life-Policy.pdf
   Premium Re

In [ ]:
# Hint: query the vectorstore for top-k results
# Hint: show retrieved chunk sources and short previews
# Hint: encourage students to try different k values


In [ ]:
# Hint: query the vectorstore for top-k results
# Hint: show retrieved chunk sources and short previews
# Hint: encourage students to try different k values


In [ ]:
# Hint: inspect the retrieved context documents (sources + content) for the last question


# Class Exercises — Try in class

### Exercise 1 — Chunking experiment
**Task:** Change chunk size and overlap, rebuild the vector store, and compare retrieved chunks for a sample question. Try at least two settings (e.g., chunk_size=800, overlap=200 and chunk_size=400, overlap=100) and note differences in which chunks are retrieved.

### Exercise 2 — Hybrid tuning (lexical + dense)
**Task:** Implement a simple fusion of TF-IDF scores and dense scores (alpha-weighted) and test alpha values [0.2, 0.5, 0.8]. Report which alpha works best for 3 sample queries.


### Exercise 3 — Re-ranking and final answer quality
**Task:** Take the top-5 candidates from your retriever and re-rank them using a Cross-Encoder (or a lightweight dense dot-product re-ranker). Compare the final answer quality (before/after re-rank) for a sample question.


## Step 6 — Generate Answer (RAG)

In [ ]:
# Hint: build a prompt that includes retrieved context + user question
# Hint: call your LLM with a short system instruction (student-friendly)
# Hint: print a concise answer and cite source filenames

## Review & Exercises

- Try adjusting chunk size and re-run retrieval
- Compare OpenAI embeddings vs. a local embedding proxy

User Query -> Retriever (Documents) -> Prompt Template -> LLM Call -> Final Answer

LangSmith Captures every step in its platform

In [ ]:
# Hint: define a simple @traceable function and call it to confirm LangSmith is capturing traces


### Evaluate


In [ ]:
# Hint: build an evaluation set - a list of questions paired with their expected/gold source document(s)


hit =  ["HDFC-Life-Group-Term-Life-Policy"]  ["HDFC-Life-Group-Term-Life-Policy"] - hit
hit =  ["HDFC-Life-Group-Term-Life-Policy"]  ["HDFC-Life-Sampoorna-Jeevan"] - no hit

Precision/ Recall

- Precision: Out of what all we retrived, how many are actually correct?

Retriever returns multiple chunks

precision = Number of relevent documents in top k/ k

In [ ]:
# Hint: write a precision@k function - retrieve top-k docs for a question, then check what fraction have a source matching the gold_sources


In [ ]:
# Hint: loop over the evaluation set, compute precision@4 for each question with your retriever
# Hint: print each question's score, then compute and print the average precision across the set


## Keyword based retriver